# Copy-state starts at oracle clones (#540)

**TL;DR.** On dev_tree 60 × 50 r0 at the planted clones: 41 starts, 1,014
trials, each polished by `sal`'s EM for up to 60 s and scored on the whole
call.

- **The lattice start is the best on BAF only and ties the best on BAF +
  RDR** (`copy_starts.lattice_start`). It places every integer `(A, B)` up
  to the rows' read-depth ceiling, as the integer decode does, assigns each
  row its most likely state under the mixture's own IID emission, fits the
  NB size and BB concentration by that likelihood, and keeps the
  `n_states` most populated.
  - BAF only: seeded on rows smoothed over 3, 5 or 9 segments it reaches the
    best fit any trial reached, in 2.1 s; on raw rows it ends 7.5 nats
    below.
  - BAF + RDR: it ends 9.2 nats below in 7.4 s, beside 10 Mb-smoothed
    `kmeans++` at 8.9.
  - Before its polish it holds the one-copy loss (log mu -0.79) and CN-LOH
    (-0.10) as two states. The polish merges them, as every fit does.
  - It is deterministic and the most robust to outliers: 0.0 nats on the 1%
    read-depth and 5% BAF corruptions, and 112 against the shortlist's 140-181
    on 5% read depth.

- **BAF + RDR: seeding on rows smoothed over 10 Mb is the best start.**
  `kmeans++` there ends 8.9 nats below the best fit reached (at most 9.3 over
  3 seeds) in 1.7 s. `--sal`'s `kmeans++x5+em` on the raw rows ends 39.6
  nats below (39.9 at most) in 13.9 s: 30 nats worse at 8x the cost. Every
  start on raw rows ends 23-49 nats below, `hmcx5` and `hmcx5+em` 152-158.
- **BAF + RDR: no start or arm separates the one-copy loss from
  copy-neutral LOH.** Across 343 fits, the LOH states (folded p < 0.1) sit
  at a median log mu of -0.12, between the loss (-0.65, 137 rows) and CN-LOH
  (0.01, 456 rows). None is within 0.1 of the loss; 15 are within 0.1 of
  CN-LOH, found in 4% of fits. The mixture objective merges the two, whatever
  the start: #471's defect.
- **BAF only: seeding from the BAF + RDR call's read-depth quantiles ends
  21.4 nats below the best, in 4.9 s.** In the pipeline the BAF stage runs
  before any normal baseline exists; against the stand-in it has, each bin's
  share of every clone's reads (`pooled_exposure`), the same seeding ends
  118.7 nats below. Port's `distinct`, today's BAF-stage
  start, ends 36.5 nats below at its median but up to 110.9 over seeds.
  Smoothing its seeding rows (any window) holds it at 36.4-38.0. One
  `emission++x5+em` trial on 10 Mb smoothing reached the best fit; its other
  two seeds were refused. BAF alone separates two levels, LOH and balanced,
  and every start finds both, so the gap measures fit quality, not state
  recovery. `sal`'s starts do not reach this stage in the pipeline yet
  (`sal_mixture` skips it).
- **Masks** on the rows a start seeds from, or fits on, do not beat
  smoothing on either stage.
- **Outliers**: 5% of rows at read depth x8 or /8 leave the shortlist
  140-181 nats below the corrupted call's best, masked or not. A mask on the
  seeding rows does not reach the polish, which still reads them.
- **Refused**:
  - `cnaster`'s `cna_mixture_init` calls `get_state_posteriors` without
    `log_sitewise_transmat`;
  - `gaussian-em` collapses a variance on BAF + RDR, as in #489;
  - `sal`'s `prior` seeds the pair `(total, fraction x total)`, the joint
    form's, and on this independent-form instance its rates exceed 1
    ("every beta must be positive"), #547;
  - `sal`'s starts refuse counts without the covariate, so covariate-free
    starts exist only as `cnaster`'s and port's GMMs;
  - on smoothed rows, `sal`'s EM refuses 25 of 72 best-of-five trials ("M
    step did not settle");
  - `emission++` draws a negative probability on 12 BAF + RDR trials.

**Seeding.** `sal_mixture.instance_of`, the instance `--sal`'s start seeds
from, seeds every BAF near 0.01 and no read-depth state below neutral
(#547). Its fit is the same model, but a start could not place a loss.
`copy_starts.instance` corrects both: exposure over 100, and the B column
over the common trial count, as `sal`'s `rate_space` writes it. An earlier
run on the uncorrected instance is discarded. These numbers are the
corrected instance's; `--sal` itself is unchanged until #547.

**Conditions.**
- The calls come from one `--sal --oracle-start` run of dev_tree r0, rebuilt
  at the planted clones for both stages (`tests.studies.copy_starts capture`).
- Every trial is a start, then `sal`'s EM on the whole call, within 60 s.
  Seeds 0-2; deterministic starts once. 4 forked workers on the 4-core host,
  one thread each, with nothing else running. Seconds include the start.
- Gap: below the best log-likelihood any trial reached on the same call.
- Found: a fitted state within 0.1 of a planted state's pooled log mu and
  0.05 of its folded p (BAF only: p alone).
- The capture is `data/copy_state_starts_capture_r0.npz`, the trials
  `data/copy_state_starts_r0.json`. Regenerate with
  `python -m tests.studies.copy_starts run docs/nb/data/copy_state_starts_capture_r0.npz OUT.pkl`,
  then `python -m tests.studies.copy_start_notebook OUT.pkl`.


In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

held = json.loads(Path("data/copy_state_starts_r0.json").read_text())
rows = pd.DataFrame(held["rows"])
settings = held["settings"]
planted = held["planted"]
print(
    f"{len(rows)} trials; {rows.error.notna().sum()} refused; "
    f"{settings['seconds']:g} s a trial, seeds {settings['seeds']}, {settings['workers']} workers"
)

1013 trials; 88 refused; 60 s a trial, seeds [0, 1, 2], 4 workers


## The planted states at oracle clones

Each planted state's pooled `(log mu, folded p, rows)` on the call: the truth a start is read against. `(0, 1)` is a one-copy loss, `(0, 2)` copy-neutral LOH, `(2, 2)` a balanced gain.

In [ ]:
pd.DataFrame(
    [
        (stage, key, *values)
        for stage, states in planted.items()
        for key, values in states.items()
    ],
    columns=["stage", "(A, B)", "log mu", "p (folded)", "rows"],
)

,stage,"(A, B)",log mu,p (folded),rows
0,baf,"0,1",0.000000,0.024533,143
1,baf,"0,2",0.000000,0.021016,466
2,baf,"1,1",0.000000,0.499774,11334
3,baf,"2,2",0.000000,0.472727,5
4,rdrbaf,"0,1",-0.652646,0.020292,137
5,rdrbaf,"0,2",0.007963,0.021305,456
6,rdrbaf,"1,1",0.000000,0.499807,10982
7,rdrbaf,"2,2",0.647769,0.472727,5


## Every start

The gap is below the best log-likelihood any trial reached on the same call, so every gap is >= 0. `found` counts the planted states some fitted state lies within 0.1 in log mu and 0.05 in folded p of (BAF-only: p alone). Median over seeds.

In [ ]:
ok = rows[rows.error.isna()].copy()
clean = ok[ok.scored_on == "call"]
best = clean.groupby("stage").log_likelihood.max()
ok["gap"] = np.where(
    ok.scored_on == "call", ok.stage.map(best) - ok.log_likelihood, np.nan
)
outliers = ok[ok.arm == "outlier"].copy()
outliers["call"] = outliers.variant.str.split(" masked").str[0]
ref = outliers.groupby(["stage", "call"]).log_likelihood.transform("max")
ok.loc[outliers.index, "gap"] = ref - outliers.log_likelihood
ok["found"] = ok.found.map(lambda f: sum(bool(v) for v in f.values()))


def table(frame):
    return (
        frame.groupby(["stage", "start"])
        .agg(
            gap=("gap", "median"),
            gap_max=("gap", "max"),
            seconds=("seconds", "median"),
            found=("found", "median"),
            trials=("seed", "size"),
        )
        .round({"gap": 1, "gap_max": 1, "seconds": 1, "found": 1})
        .sort_values(["stage", "gap"])
    )


starts = ok[ok.arm == "starts"]
table(starts)

gap  gap_max  seconds  found  trials
stage  start                                               
baf    lattice         7.5      7.5      1.7    4.0       1
       distinct       40.4    114.8      5.5    4.0       3
       emission++     47.0     69.1      1.4    4.0       3
       emission++x5   49.8     55.3      3.1    4.0       3
       kmeans++       59.1    107.1      2.5    4.0       3
...                    ...      ...      ...    ...     ...
rdrbaf objective      48.6     48.6      1.8    1.0       1
       quantile       48.6     48.6      1.6    1.0       1
       perturbed      48.7     48.7      1.9    1.0       1
       hmcx5         151.7    274.6     84.9    1.0       3
       hmcx5+em      157.8    178.7     84.8    1.0       3

[72 rows x 5 columns]

In [ ]:
refused = rows[rows.error.notna()]
refused.assign(error=refused.error.str.slice(0, 90)).groupby(
    ["arm", "stage", "start"]
).error.first().to_frame()

error
arm           stage  start                                                              
mask-fit      rdrbaf emission++x5+em      ValueError: Probabilities are not non-negative
mask-seed     rdrbaf emission++x5+em      ValueError: Probabilities are not non-negative
no-covariate  baf    datax5+em         ParameterDomainError: every beta must be posit...
                     emission++x5+em               ValueError: Probabilities contain NaN
                     kmeans++          ParameterDomainError: every beta must be posit...
                     kmeans++x5+em     ParameterDomainError: every beta must be posit...
              rdrbaf datax5+em         ParameterDomainError: every beta must be posit...
                     emission++x5+em               ValueError: Probabilities contain NaN
                     kmeans++          ParameterDomainError: every beta must be posit...
                     kmeans++x5+em     ParameterDomainError: every beta must be posit...
outlier       rdrbaf emission++x5+em      ValueError: Probabilities are not non-negative
rdr-seeds-baf baf    emission++x5+em      ValueError: Probabilities are not non-negative
smooth        baf    emission++x5+em   ValueError: a component's M step did not settl...
                     kmeans++x5+em     ValueError: a component's M step did not settl...
              rdrbaf emission++x5+em   ValueError: a component's M step did not settl...
                     kmeans++x5+em     ValueError: a component's M step did not settl...
starts        baf    prior             ParameterDomainError: every beta must be posit...
                     priorx5           ParameterDomainError: every beta must be posit...
                     priorx5+em        ParameterDomainError: every beta must be posit...
              rdrbaf cna-mixture++     TypeError: hmm_nophasing.get_state_posteriors(...
                     emission++x5         ValueError: Probabilities are not non-negative
                     emission++x5+em      ValueError: Probabilities are not non-negative
                     gaussian-em       ValueError: state(s) [2] re-estimated to varia...
                     gaussian-emx5     ValueError: state(s) [2] re-estimated to varia...
                     gaussian-emx5+em  ValueError: state(s) [2] re-estimated to varia...
                     priorx5           ParameterDomainError: every beta must be posit...
                     priorx5+em        ParameterDomainError: every beta must be posit...

## Runtime against gap

Each start's median over seeds, with the seeds' range; one panel per stage. Named: the six nearest the best fit and the pipeline's own starts; the rest grey. Starred: the three best arms that seed on masked, smoothed or RDR-derived rows. The deprecated `docs/plots/studies/hmm_starts.png` (#489) drew the BAF + RDR stage alone.

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from port.extensions.figure_style import figure_rc

mpl.use("Agg")

with mpl.rc_context(figure_rc()):
    figure, axes = plt.subplots(1, 2, figsize=(10, 4.2), sharey=False)
    for ax, (stage, title) in zip(
        axes, [("baf", "BAF only"), ("rdrbaf", "BAF + RDR")], strict=True
    ):
        frame = starts[starts.stage == stage]
        summary = frame.groupby("start").agg(
            gap=("gap", "median"),
            lo=("gap", "min"),
            hi=("gap", "max"),
            seconds=("seconds", "median"),
            found=("found", "median"),
        )
        floor = 0.1
        # NB named: the six nearest the best fit, and the starts the pipeline
        #    installs (`distinct`, `cnaster-gmm`, `--sal`'s `kmeans++x5+em`);
        #    the rest grey, unnamed.
        named = set(summary.gap.nsmallest(6).index) | {
            "distinct",
            "cnaster-gmm",
            "kmeans++x5+em",
        }
        for name, row in summary.iterrows():
            y = max(row.gap, floor)
            shown = name in named
            ax.errorbar(
                row.seconds,
                y,
                yerr=[[y - max(row.lo, floor)], [max(row.hi, floor) - y]],
                fmt="o",
                ms=4 if shown else 3,
                capsize=2,
                alpha=0.9 if shown else 0.35,
                color=None if shown else "0.6",
                zorder=3 if shown else 1,
            )
            if shown:
                ax.annotate(
                    name,
                    (row.seconds, y),
                    fontsize=7,
                    xytext=(4, 2),
                    textcoords="offset points",
                )
        # NB starred: the three best arms other than `starts` on this stage,
        #    each a start seeded on masked, smoothed or RDR-derived rows.
        armed = ok[
            (ok.stage == stage) & (ok.arm != "starts") & (ok.scored_on == "call")
        ]
        tops = (
            armed.groupby(["arm", "variant", "start"])
            .agg(gap=("gap", "median"), seconds=("seconds", "median"))
            .nsmallest(3, "gap")
        )
        for (arm, variant, name), row in tops.iterrows():
            y = max(row.gap, floor)
            ax.plot(row.seconds, y, marker="*", ms=9, color="black", zorder=4)
            ax.annotate(
                f"{name}, {variant or arm}",
                (row.seconds, y),
                fontsize=7,
                xytext=(4, -9),
                textcoords="offset points",
            )
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_xlabel("seconds, start and polish")
        ax.set_title(f"{title}: {len(summary)} starts", loc="left")
    axes[0].set_ylabel("gap below the best fit reached [nats] (0 drawn at 0.1)")
    figure.tight_layout()
    figure.savefig(
        "../plots/studies/copy_state_starts.png", dpi=150, metadata={"Software": None}
    )
    plt.show()

## Arms on the shortlist

Each arm against the same start's own `starts` row: the change in median gap (negative is better) and in states found. `mask-seed` masks the rows a start seeds from; `mask-fit` also fits on them; `smooth` seeds on rows summed along the genome; every result is polished and scored on the whole call.

In [ ]:
base = table(starts)[["gap", "found"]]
arms = ok[~ok.arm.isin(["starts", "outlier"])]
by = arms.groupby(["arm", "variant", "stage", "start"]).agg(
    gap=("gap", "median"), found=("found", "median"), seconds=("seconds", "median")
)
joined = by.join(base, on=["stage", "start"], rsuffix="_start")
joined["d_gap"] = (joined.gap - joined.gap_start).round(1)
joined["d_found"] = joined.found - joined.found_start
joined[["gap", "d_gap", "found", "d_found", "seconds"]].round(1)

gap  d_gap  found  d_found  \
arm      variant    stage  start                                          
mask-fit baf-se-0.1 baf    datax5+em        69.6    6.6    4.0      0.0   
                           distinct         62.1   21.7    4.0      0.0   
                           emission++x5+em  69.6    0.0    4.0      0.0   
                           kmeans++         69.6   10.5    4.0      0.0   
                           kmeans++x5+em    69.6   -0.2    4.0      0.0   
...                                          ...    ...    ...      ...   
smooth   9 segments rdrbaf datax5+em        38.9   -0.4    1.0      0.0   
                           distinct         38.4    0.0    1.0      0.0   
                           kmeans++          9.7  -25.7    1.0      0.0   
                           lattice           9.3    0.1    1.0      0.0   
                           rdr-quantiles    38.9   -0.1    1.0      0.0   

                                            seconds  
arm      variant    stage  start                     
mask-fit baf-se-0.1 baf    datax5+em           10.9  
                           distinct             5.8  
                           emission++x5+em     12.7  
                           kmeans++             3.6  
                           kmeans++x5+em       12.0  
...                                             ...  
smooth   9 segments rdrbaf datax5+em           19.6  
                           distinct             3.8  
                           kmeans++             4.0  
                           lattice              4.0  
                           rdr-quantiles        1.9  

[226 rows x 5 columns]

## Outliers

The shortlist on calls with 1% or 5% of rows replaced (read depth x8 or /8; B at 0 or at its trials), masked and not. The gap is on the corrupted call; `found` is against the clean call's states.

In [ ]:
(
    ok[ok.arm == "outlier"]
    .groupby(["stage", "variant", "start"])
    .agg(
        gap=("gap", "median"), found=("found", "median"), seconds=("seconds", "median")
    )
    .round(1)
)

gap  found  seconds
stage  variant                  start                                 
baf    baf 1%                   datax5+em         75.1    2.0      7.1
                                distinct          72.6    2.0      2.1
                                emission++x5+em   43.9    4.0     14.6
                                kmeans++          49.8    4.0      0.9
                                kmeans++x5+em     62.1    4.0      8.4
...                                                ...    ...      ...
rdrbaf rdr 5% masked rdr-top-5% emission++x5+em    0.1    1.0     19.6
                                kmeans++           0.1    1.0      6.3
                                kmeans++x5+em    374.5    1.0     12.9
                                lattice          138.2    1.0      7.1
                                rdr-quantiles    395.2    2.0      1.6

[84 rows x 3 columns]